# 11 — Explicabilidade: análise por camada, early exit e custo computacional

Notebook **autocontido** que reproduz os experimentos de explicabilidade da dissertação
(e do artigo *"Information, Behavior, and Format: Where Syntax Lives in BERT Depends on
Who Is Asking"*) sobre os modelos ParseH2IA treinados no corpus Porttinari:

| § | Experimento | O que mede | Custo aprox. (RTX 4090) |
|---|---|---|---|
| 2 | **Logit lens** por camada | comportamento: o que as cabeças finais extraem de cada profundidade | ~2 min/modelo |
| 3 | **Early exit** (ablação causal) | truncar o encoder na camada $k$ e ler todas as tarefas dali | offline (reusa §2) |
| 4 | **Block skip** | embeddings alimentadas só em subconjuntos de blocos | ~8 min |
| 5 | Variante **biaffine** de §2–3 | o mesmo perfil sob decoder relacional | ~3 min |
| 6 | **Probes lineares** por camada | informação linearmente decodificável (dissociação de formato) | ~15 min |
| 7 | **Early exit implantável de UPOS** | encoder fisicamente truncado + cabeça linear; latência real | ~10 min |
| 8 | Cabeças congeladas (saturação) | ponteiro para `scripts/layerwise_frozen_heads.py` | — |

**Pré-requisitos** (não versionados neste repositório — ver README):
- checkpoints `linear_BERTimbau_base/` e `biaffine_BERTImbau_base/checkpoint-24321/`;
- corpus Porttinari processado em `data_dois/complaints_dataset_obj_outxpos`
  (formato HuggingFace `datasets`).

Por padrão assume-se que o repositório está clonado dentro do diretório de dados
(`PARSEH2IA_DATA`, default `../..`). Os CSVs produzidos aqui reproduzem os de
`../artifacts/layerwise/`.

## 0. Configuração

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Diretório com checkpoints e dataset (fora do repositório)
DATA = os.environ.get('PARSEH2IA_DATA', '../..')

LINEAR_MODEL_PATH   = f'{DATA}/linear_BERTimbau_base'
BIAFFINE_MODEL_PATH = f'{DATA}/biaffine_BERTImbau_base/checkpoint-24321'
TOKENIZER_NAME      = 'neuralmind/bert-base-portuguese-cased'   # BERTimbau-base
DATASET_PATH        = f'{DATA}/data_dois/complaints_dataset_obj_outxpos'
OUT_DIR             = './outputs_explicabilidade'
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'


seed_everything(42)
print(f'device: {DEVICE}')

In [ ]:
# Vocabulários de labels do dataset de treino (ordenados por frequência)
UPOS_LABELS = ['DET', 'NOUN', 'VERB', 'PUNCT', 'SCONJ', 'ADP', 'ADJ', 'CCONJ', 'ADV',
               'PROPN', 'AUX', 'NUM', 'PRON', 'SYM', 'X', 'INTJ']
DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case',
                 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop',
                 'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj',
                 'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer',
                 'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated',
                 'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list',
                 'reparandum', 'csubj:pass']

UPOS_LABELS_TO_IDX   = {l: i for i, l in enumerate(UPOS_LABELS)}
IDX_TO_UPOS_LABELS   = {i: l for l, i in UPOS_LABELS_TO_IDX.items()}
DEPREL_LABELS_TO_IDX = {l: i for i, l in enumerate(DEPREL_LABELS)}
IDX_TO_DEPREL_LABELS = {i: l for l, i in DEPREL_LABELS_TO_IDX.items()}

# Cores fixas por tarefa — mantidas em TODOS os gráficos do notebook
TASK_COLORS = {'upos': 'tab:blue', 'deprel': 'tab:orange', 'head': 'tab:green'}

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk(DATASET_PATH)
train_dataset, test_dataset = dataset['train'], dataset['test']

test_sentences = test_dataset['tokens']
test_upos      = test_dataset['upos']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f'train: {len(train_dataset)} sentenças | test: {len(test_sentences)} sentenças')

## 1. Definição e carregamento dos modelos

As duas famílias de modelo da dissertação, com os mesmos pesos dos checkpoints avaliados
nos notebooks 06–07:

- **Linear MTL**: classificadores lineares por token para UPOS, DEPREL e HEAD
  (HEAD como classificação posicional sobre 200 posições);
- **Biaffine MTL** (Dozat & Manning, 2017): UPOS linear + MLPs arc/rel + scorers
  biaffine sobre pares de tokens; DEPREL avaliado no arco predito.

In [ ]:
from transformers import PreTrainedModel, AutoModel, AutoConfig, AutoTokenizer


class LinearMTLModel(PreTrainedModel):
    """MTL com cabeçotes lineares por token (HEAD posicional, 200 classes)."""
    _tied_weights_keys = []
    all_tied_weights_keys = {}

    def __init__(self, config, num_deprel_labels, num_upos_labels, num_head_labels=200):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels
        self.num_head_labels = num_head_labels
        self.bert = AutoModel.from_config(config)

        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
        self.head_classifier = nn.Linear(config.hidden_size, num_head_labels)

        classifier_dropout = (config.classifier_dropout
                              if config.classifier_dropout is not None
                              else config.hidden_dropout_prob)
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, upos_label=None, head_label=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        seq = self.dropout(outputs[0])

        logits_deprel = self.deprel_classifier(seq)
        logits_upos = self.upos_classifier(seq)
        logits_head = self.head_classifier(seq)

        loss = None
        if deprel_label is not None and upos_label is not None and head_label is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = (loss_fct(logits_deprel.view(-1, self.num_deprel_labels), deprel_label.view(-1))
                    + loss_fct(logits_upos.view(-1, self.num_upos_labels), upos_label.view(-1))
                    + loss_fct(logits_head.view(-1, self.num_head_labels), head_label.view(-1)))

        return ((loss, logits_deprel, logits_upos, logits_head) if loss is not None
                else (logits_deprel, logits_upos, logits_head))

In [ ]:
from typing import Optional

from transformers import BertModel, BertPreTrainedModel


class MLP(nn.Module):
    """Projeção não-linear usada antes das camadas biaffine."""

    def __init__(self, in_features: int, out_features: int, dropout: float = 0.33):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ELU()
        self.norm = nn.LayerNorm(out_features)
        self.dropout = nn.Dropout(dropout)
        nn.init.orthogonal_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return self.dropout(self.norm(self.activation(self.linear(x))))


class Biaffine(nn.Module):
    """score[b, o, i, j] = x[b,i]ᵀ · W[o] · y[b,j]  (Dozat & Manning, 2017).

    x → dependente; y → head candidato.
    Arc: out_features=1, bias_x=True, bias_y=False.
    Rel: out_features=num_deprel, bias_x=True, bias_y=True.
    """

    def __init__(self, in_features, out_features=1, bias_x=True, bias_y=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bias_x, self.bias_y = bias_x, bias_y
        self.weight = nn.Parameter(torch.zeros(
            out_features, in_features + int(bias_x), in_features + int(bias_y)))
        nn.init.normal_(self.weight, std=1.0 / in_features)

    def forward(self, x, y):
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)
        return torch.einsum('bih,ohk,bjk->boij', x, self.weight, y)


class BiaffineMTLModel(BertPreTrainedModel):
    """MTL biaffine: UPOS linear + arc/rel MLPs + scorers biaffine."""

    def __init__(self, config, num_deprel_labels, num_upos_labels,
                 arc_hidden=500, rel_hidden=100, mlp_dropout=0.33):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels
        self.arc_hidden, self.rel_hidden = arc_hidden, rel_hidden

        self.bert = BertModel(config, add_pooling_layer=False)
        encoder_dropout = (config.classifier_dropout
                           if getattr(config, 'classifier_dropout', None) is not None
                           else config.hidden_dropout_prob)
        self.dropout = nn.Dropout(encoder_dropout)

        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.arc_biaffine = Biaffine(arc_hidden, out_features=1, bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels,
                                     bias_x=True, bias_y=True)
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, upos_label=None, head_label=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        seq = self.dropout(outputs.last_hidden_state)
        B, L, _ = seq.shape

        logits_upos = self.upos_classifier(seq)
        logits_head = self.arc_biaffine(self.arc_dep_mlp(seq), self.arc_head_mlp(seq)).squeeze(1)
        if attention_mask is not None:
            logits_head = logits_head.masked_fill((attention_mask == 0).unsqueeze(1), -1e4)
        logits_rel = self.rel_biaffine(self.rel_dep_mlp(seq), self.rel_head_mlp(seq))

        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)
        idx = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_deprel_out = (logits_rel.permute(0, 2, 3, 1).contiguous()
                             .gather(2, idx).squeeze(2))

        return (logits_deprel_out, logits_upos, logits_head)

In [ ]:
linear_config = AutoConfig.from_pretrained(LINEAR_MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

linear_model = LinearMTLModel.from_pretrained(
    LINEAR_MODEL_PATH, config=linear_config,
    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS),
).to(DEVICE).eval()

biaffine_model = BiaffineMTLModel.from_pretrained(
    BIAFFINE_MODEL_PATH,
    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS),
).to(DEVICE).eval()

N_LAYERS = linear_config.num_hidden_layers          # 12 no BERTimbau-base
print(f'linear:   {sum(p.numel() for p in linear_model.parameters())/1e6:.1f}M params')
print(f'biaffine: {sum(p.numel() for p in biaffine_model.parameters())/1e6:.1f}M params')

## 2. Logit lens: acurácia por camada com as cabeças finais

As cabeças **treinadas na camada final** são aplicadas aos estados ocultos de cada camada
$\ell \in \{0, \dots, 12\}$ ($\ell = 0$ = embeddings). Isso mede o *comportamento*: o que
o modelo implantado consegue extrair de cada profundidade. Como as cabeças são calibradas
para a camada 12, os valores em camadas intermediárias são **limites inferiores** da
informação presente (as sondas da §6 apertam esse limite).

Uma única passada pelo test set guarda o argmax de cada camada para cada tarefa,
permitindo avaliar offline qualquer combinação de camadas (§3).

In [ ]:
def collect_predictions_all_layers(sentences, gold_upos_list, gold_deprel_list,
                                   gold_head_list, model, tokenizer, max_sentences=None):
    """Uma passada: argmax de CADA camada para CADA tarefa (cabeças lineares finais)."""
    device = next(model.parameters()).device
    model.eval()
    classifiers = {'upos': model.upos_classifier, 'deprel': model.deprel_classifier,
                   'head': model.head_classifier}
    preds_store, golds_store = [], []

    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n)):
        tokens = sentences[i]
        golds = {'upos':   [UPOS_LABELS_TO_IDX[u] for u in gold_upos_list[i]],
                 'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
                 'head':   [int(h) for h in gold_head_list[i]]}
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors='pt',
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(input_ids=inputs['input_ids'],
                                  attention_mask=inputs['attention_mask'],
                                  token_type_ids=inputs.get('token_type_ids'),
                                  output_hidden_states=True)
            hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]   # [13, seq, H]

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue
            hs_tok = hs_all[:, torch.tensor(first_subs, device=device)]

            sent_preds = {t: clf(hs_tok).argmax(-1).cpu().numpy().astype(np.int16)
                          for t, clf in classifiers.items()}
            sent_golds = {t: np.array([golds[t][k] for k in tok_idxs], dtype=np.int16)
                          for t in classifiers}
            preds_store.append(sent_preds)
            golds_store.append(sent_golds)
    return preds_store, golds_store


def eval_layer_combo(preds_store, golds_store, upos_layer, deprel_layer, head_layer):
    """Acurácias + UAS/LAS para uma combinação de camadas (uma por tarefa)."""
    total = upos_c = deprel_c = uas_c = las_c = 0
    for preds, golds in zip(preds_store, golds_store):
        p_upos, p_deprel, p_head = (preds['upos'][upos_layer], preds['deprel'][deprel_layer],
                                    preds['head'][head_layer])
        g_upos, g_deprel, g_head = golds['upos'], golds['deprel'], golds['head']
        total += len(g_upos)
        upos_c += (p_upos == g_upos).sum()
        deprel_c += (p_deprel == g_deprel).sum()
        head_ok = (p_head == g_head)
        uas_c += head_ok.sum()
        las_c += (head_ok & (p_deprel == g_deprel)).sum()
    return {'upos_layer': upos_layer, 'deprel_layer': deprel_layer, 'head_layer': head_layer,
            'upos_acc': upos_c / total, 'deprel_acc': deprel_c / total,
            'uas': uas_c / total, 'las': las_c / total}

In [ ]:
# Passada única no test set completo (~2 min em GPU)
preds_lin, golds_lin = collect_predictions_all_layers(
    test_sentences, test_upos, test_deprel, test_head, linear_model, tokenizer)

n_layers_total = preds_lin[0]['upos'].shape[0]           # 13 (embeddings + 12)
per_layer_lin = pd.DataFrame([
    eval_layer_combo(preds_lin, golds_lin, L, L, L) for L in range(n_layers_total)
]).rename(columns={'upos_layer': 'layer'}).drop(columns=['deprel_layer', 'head_layer'])
per_layer_lin.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
for metric, task, label in (('upos_acc', 'upos', 'UPOS (acurácia)'),
                            ('deprel_acc', 'deprel', 'DEPREL (acurácia)'),
                            ('uas', 'head', 'HEAD (UAS)')):
    ax.plot(per_layer_lin['layer'], per_layer_lin[metric],
            color=TASK_COLORS[task], linewidth=2, marker='o', markersize=5, label=label)
ax.set_xlabel('Camada (0 = embeddings)')
ax.set_ylabel('Acurácia no test set')
ax.set_title('Logit lens — modelo Linear MTL (BERTimbau-base)')
ax.set_xticks(range(n_layers_total))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# A hierarquia UPOS → DEPREL → HEAD em profundidade é o achado central:
# UPOS aproxima-se do teto por volta das camadas 8–10; HEAD só resolve no topo.

## 3. Early exit: truncar o encoder na camada $k$

Ablação causal: todas as tarefas são lidas da camada $k$, como se o encoder fosse
interrompido ali. O resultado replica `artifacts/layerwise/layerwise_early_exit_comparison.csv`:
truncar na camada 10 custa <0,5 p.p. de tagging, mas **~49 pontos de UAS** — a estrutura
de dependências é composta pelas camadas superiores e não sobrevive ao truncamento.

In [ ]:
FINAL = n_layers_total - 1        # camada 12
BEST_LAYER = {'upos':   int(per_layer_lin['upos_acc'].idxmax()),
              'deprel': int(per_layer_lin['deprel_acc'].idxmax()),
              'head':   int(per_layer_lin['uas'].idxmax())}
print('Melhor camada por tarefa:', BEST_LAYER)

configs = {
    'baseline: tudo na camada final (12)': (FINAL, FINAL, FINAL),
    'melhor camada por tarefa':            (BEST_LAYER['upos'], BEST_LAYER['deprel'], BEST_LAYER['head']),
    'camadas de convergência (8, 9, 12)':  (8, 9, FINAL),
    'early exit na camada 10':             (10, 10, 10),
    'early exit na camada 9':              (9, 9, 9),
    'early exit na camada 8':              (8, 8, 8),
}
rows = []
for name, (lu, ld, lh) in configs.items():
    r = eval_layer_combo(preds_lin, golds_lin, lu, ld, lh)
    r['config'] = name
    rows.append(r)
early_exit_lin = pd.DataFrame(rows)[
    ['config', 'upos_layer', 'deprel_layer', 'head_layer', 'upos_acc', 'deprel_acc', 'uas', 'las']]
early_exit_lin.to_csv(f'{OUT_DIR}/layerwise_early_exit_comparison.csv', index=False)
early_exit_lin.round(4)

## 4. Block skip: quais blocos são substituíveis?

As embeddings são alimentadas **apenas** nos blocos indicados. Se as camadas superiores
apenas refinassem uma solução já formada ("refinamento iterativo"), o último bloco sozinho
— ou iterado 12× para igualar os FLOPs da rede completa — deveria recuperar parte do
desempenho. O resultado rejeita essa hipótese: profundidade é uma **cadeia composicional**.

In [ ]:
def eval_block_subset(blocks, sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                      model, tokenizer, max_sentences=None, desc=None):
    """Passa as embeddings APENAS pelos blocos indicados (0-based; bloco 11 = camada 12)."""
    device = next(model.parameters()).device
    model.eval()
    encoder_layers = model.bert.encoder.layer
    total = upos_c = deprel_c = uas_c = las_c = 0

    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n), desc=desc):
        tokens = sentences[i]
        golds = {'upos':   np.array([UPOS_LABELS_TO_IDX[u] for u in gold_upos_list[i]]),
                 'deprel': np.array([DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]]),
                 'head':   np.array([int(h) for h in gold_head_list[i]])}
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors='pt',
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            hs = model.bert.embeddings(input_ids=inputs['input_ids'],
                                       token_type_ids=inputs.get('token_type_ids'))
            for b in blocks:                       # batch=1 sem padding → sem máscara
                hs = encoder_layers[b](hs)
                if isinstance(hs, tuple):
                    hs = hs[0]

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue
            hs_tok = hs[0, torch.tensor(first_subs, device=device)]
            p_upos = model.upos_classifier(hs_tok).argmax(-1).cpu().numpy()
            p_deprel = model.deprel_classifier(hs_tok).argmax(-1).cpu().numpy()
            p_head = model.head_classifier(hs_tok).argmax(-1).cpu().numpy()

        g_upos, g_deprel, g_head = (golds[t][tok_idxs] for t in ('upos', 'deprel', 'head'))
        total += len(tok_idxs)
        upos_c += (p_upos == g_upos).sum()
        deprel_c += (p_deprel == g_deprel).sum()
        head_ok = (p_head == g_head)
        uas_c += head_ok.sum()
        las_c += (head_ok & (p_deprel == g_deprel)).sum()

    blocos = ','.join(str(b + 1) for b in blocks) if blocks else 'nenhum (embeddings)'
    return {'blocos_ativos': blocos, 'n_blocos': len(blocks),
            'upos_acc': upos_c / total, 'deprel_acc': deprel_c / total,
            'uas': uas_c / total, 'las': las_c / total}


block_configs = {
    'sanity: todos os blocos (== inferência normal)': list(range(12)),
    'APENAS bloco 12':                                [11],
    'APENAS bloco 12, aplicado 12x':                  [11] * 12,
    'blocos 11 e 12':                                 [10, 11],
    'blocos 9 a 12':                                  [8, 9, 10, 11],
    'blocos 7 a 12 (pula metade inferior)':           [6, 7, 8, 9, 10, 11],
    'nenhum bloco (embeddings puras)':                [],
}
block_rows = []
for name, blocks in block_configs.items():
    r = eval_block_subset(blocks, test_sentences, test_upos, test_deprel, test_head,
                          linear_model, tokenizer, desc=name)
    r['config'] = name
    block_rows.append(r)
block_comparison = pd.DataFrame(block_rows)[
    ['config', 'blocos_ativos', 'n_blocos', 'upos_acc', 'deprel_acc', 'uas', 'las']]
block_comparison.to_csv(f'{OUT_DIR}/layerwise_block_skip_comparison.csv', index=False)
block_comparison.round(4)

## 5. Variante biaffine: o mesmo perfil sob decoder relacional

O logit lens biaffine aplica os MLPs + scorers biaffine por camada, lendo DEPREL no arco
predito **pela própria camada** (mesmo protocolo do `forward`). O biaffine converge mais
cedo (UPOS ~6 vs. ~8 do linear), mas o perfil de profundidade de HEAD é o mesmo: resolução
no quarto superior da rede.

In [ ]:
def biaffine_heads_from_hidden(model, hs):
    """Aplica as cabeças biaffine treinadas sobre estados ocultos [N, L, H]."""
    logits_upos = model.upos_classifier(hs)
    logits_head = model.arc_biaffine(model.arc_dep_mlp(hs), model.arc_head_mlp(hs)).squeeze(1)
    logits_rel = model.rel_biaffine(model.rel_dep_mlp(hs), model.rel_head_mlp(hs))

    N, L, _ = logits_head.shape
    arc_preds = logits_head.argmax(-1).clamp(0, L - 1)
    idx = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(N, L, 1, model.num_deprel_labels)
    logits_deprel = (logits_rel.permute(0, 2, 3, 1).contiguous()
                     .gather(2, idx).squeeze(2))
    return logits_upos, logits_head, logits_deprel


def collect_predictions_all_layers_biaffine(sentences, gold_upos_list, gold_deprel_list,
                                            gold_head_list, model, tokenizer,
                                            max_sentences=None):
    device = next(model.parameters()).device
    model.eval()
    preds_store, golds_store = [], []
    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n)):
        tokens = sentences[i]
        golds = {'upos':   [UPOS_LABELS_TO_IDX[u] for u in gold_upos_list[i]],
                 'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
                 'head':   [int(h) for h in gold_head_list[i]]}
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors='pt',
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(input_ids=inputs['input_ids'],
                                  attention_mask=inputs['attention_mask'],
                                  token_type_ids=inputs.get('token_type_ids'),
                                  output_hidden_states=True)
            hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]
            logits_upos, logits_head, logits_deprel = biaffine_heads_from_hidden(model, hs_all)

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue
            fs = torch.tensor(first_subs, device=device)
            preds_store.append({
                'upos':   logits_upos[:, fs].argmax(-1).cpu().numpy().astype(np.int16),
                'deprel': logits_deprel[:, fs].argmax(-1).cpu().numpy().astype(np.int16),
                'head':   logits_head[:, fs].argmax(-1).cpu().numpy().astype(np.int16)})
            golds_store.append({t: np.array([golds[t][k] for k in tok_idxs], dtype=np.int16)
                                for t in ('upos', 'deprel', 'head')})
    return preds_store, golds_store

In [ ]:
preds_bia, golds_bia = collect_predictions_all_layers_biaffine(
    test_sentences, test_upos, test_deprel, test_head, biaffine_model, tokenizer)

per_layer_bia = pd.DataFrame([
    eval_layer_combo(preds_bia, golds_bia, L, L, L) for L in range(n_layers_total)
]).rename(columns={'upos_layer': 'layer'}).drop(columns=['deprel_layer', 'head_layer'])
per_layer_bia.to_csv(f'{OUT_DIR}/layerwise_per_layer_metrics_biaffine.csv', index=False)

rows = []
for name, (lu, ld, lh) in {
        'baseline: tudo na camada final (12)': (FINAL, FINAL, FINAL),
        'early exit na camada 11': (11, 11, 11),
        'early exit na camada 10': (10, 10, 10),
        'early exit na camada 9':  (9, 9, 9),
        'early exit na camada 8':  (8, 8, 8)}.items():
    r = eval_layer_combo(preds_bia, golds_bia, lu, ld, lh)
    r['config'] = name
    rows.append(r)
early_exit_bia = pd.DataFrame(rows)[
    ['config', 'upos_layer', 'deprel_layer', 'head_layer', 'upos_acc', 'deprel_acc', 'uas', 'las']]
early_exit_bia.to_csv(f'{OUT_DIR}/layerwise_early_exit_comparison_biaffine.csv', index=False)
early_exit_bia.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(per_layer_lin['layer'], per_layer_lin['las'], color='#4c72b0', linewidth=2,
        marker='o', markersize=5, label='Linear MTL')
ax.plot(per_layer_bia['layer'], per_layer_bia['las'], color='#dd8452', linewidth=2,
        marker='s', markersize=5, label='Biaffine MTL')
ax.set_xlabel('Camada (0 = embeddings)')
ax.set_ylabel('LAS no test set')
ax.set_title('Logit lens — LAS por camada: Linear vs. Biaffine (BERTimbau-base)')
ax.set_xticks(range(n_layers_total))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Probes lineares por camada: informação vs. comportamento vs. formato

Em vez de ler as camadas com a cabeça da camada final (limite inferior), treinamos um
**probe linear por camada** sobre o encoder congelado. Isso mede a informação linearmente
decodificável em cada profundidade.

Dois achados a reproduzir:
1. **Informação satura cedo**: UPOS atinge ~98% na camada 3 (o logit lens da §2 só
   converge nas camadas 6–9 — a informação existe antes de a cabeça final conseguir lê-la);
2. **Dissociação de formato**: probes posicionais de HEAD chegam a ~94% no encoder do
   modelo *linear*, mas ficam em ~26% no encoder *biaffine* em TODAS as camadas — apesar
   de o parser biaffine acertar ~90% dos arcos. O decoder dita o formato representacional;
   probing cego a formato declararia um encoder de 90 UAS "sem sintaxe".

Protocolo do artigo: 3.000 sentenças de treino, AdamW lr=1e-3, 30 épocas.
(No artigo são 5 seeds; aqui o default são 3 para reduzir o tempo — σ ≤ 0,003.)

In [ ]:
N_TRAIN_SENTS = 3000
PROBE_SEEDS = (0, 1, 2)
PROBE_EPOCHS = 30
PROBE_BATCH = 8192


def extract_hidden_states(encoder, tokenizer, sentences, upos, deprel, head):
    """Estados ocultos (todas as camadas, fp16, CPU) no 1º subtoken de cada palavra."""
    feats, golds = [], {'upos': [], 'deprel': [], 'head': []}
    encoder.eval()
    for i in tqdm(range(len(sentences)), desc='extract'):
        tokens = sentences[i]
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors='pt',
                           padding=True, truncation=True).to(DEVICE)
        with torch.no_grad():
            out = encoder(input_ids=inputs['input_ids'],
                          attention_mask=inputs['attention_mask'],
                          token_type_ids=inputs.get('token_type_ids'),
                          output_hidden_states=True)
            hs = torch.stack(out.hidden_states, dim=0)[:, 0]

        word_ids = inputs.word_ids(batch_index=0)
        first_subs, tok_idxs = [], []
        seen = set()
        for pos, w in enumerate(word_ids):
            if w is not None and w not in seen and w < len(tokens):
                seen.add(w)
                first_subs.append(pos)
                tok_idxs.append(w)
        if not first_subs:
            continue
        feats.append(hs[:, torch.tensor(first_subs, device=DEVICE)].half().cpu())
        golds['upos'].append(np.array([UPOS_LABELS_TO_IDX[upos[i][t]] for t in tok_idxs]))
        golds['deprel'].append(np.array([DEPREL_LABELS_TO_IDX[deprel[i][t]] for t in tok_idxs]))
        golds['head'].append(np.array([int(head[i][t]) for t in tok_idxs]))

    X = torch.cat(feats, dim=1)                       # [n_layers, N, H]
    y = {t: torch.tensor(np.concatenate(golds[t]), dtype=torch.long) for t in golds}
    return X, y


def train_probe(X_tr, y_tr, X_te, y_te, n_classes, seed):
    torch.manual_seed(seed)
    probe = nn.Linear(X_tr.shape[1], n_classes).to(DEVICE)
    opt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fct = nn.CrossEntropyLoss()
    n = X_tr.shape[0]
    g = torch.Generator().manual_seed(seed)
    for _ in range(PROBE_EPOCHS):
        perm = torch.randperm(n, generator=g)
        for s in range(0, n, PROBE_BATCH):
            idx = perm[s:s + PROBE_BATCH]
            opt.zero_grad()
            loss = loss_fct(probe(X_tr[idx]), y_tr[idx])
            loss.backward()
            opt.step()
    probe.eval()
    with torch.no_grad():
        preds = torch.cat([probe(X_te[s:s + PROBE_BATCH]).argmax(-1)
                           for s in range(0, X_te.shape[0], PROBE_BATCH)])
    return (preds == y_te).float().mean().item()

In [ ]:
rng = np.random.default_rng(42)
tr_idx = rng.choice(len(train_dataset), size=N_TRAIN_SENTS, replace=False)
probe_train = train_dataset.select(tr_idx)

probe_rows = []
for enc_name, mtl_model in (('lin_bertimbau', linear_model),
                            ('bia_bertimbau', biaffine_model)):
    print(f'════ {enc_name} ════')
    encoder = mtl_model.bert                     # encoder fine-tunado, congelado
    X_tr, y_tr = extract_hidden_states(encoder, tokenizer, probe_train['tokens'],
                                       probe_train['upos'], probe_train['deprel'],
                                       probe_train['head_tags'])
    X_te, y_te = extract_hidden_states(encoder, tokenizer, test_sentences,
                                       test_upos, test_deprel, test_head)
    n_head_classes = int(max(y_tr['head'].max(), y_te['head'].max())) + 1
    n_classes = {'upos': len(UPOS_LABELS), 'deprel': len(DEPREL_LABELS),
                 'head': n_head_classes}

    for layer in tqdm(range(X_tr.shape[0]), desc='probes'):
        Xl_tr = X_tr[layer].float().to(DEVICE)
        Xl_te = X_te[layer].float().to(DEVICE)
        for task in ('upos', 'deprel', 'head'):
            for seed in PROBE_SEEDS:
                acc = train_probe(Xl_tr, y_tr[task].to(DEVICE), Xl_te, y_te[task].to(DEVICE),
                                  n_classes[task], seed)
                probe_rows.append({'encoder': enc_name, 'layer': layer,
                                   'task': task, 'seed': seed, 'acc': acc})
        del Xl_tr, Xl_te
        torch.cuda.empty_cache()
    del X_tr, X_te
    pd.DataFrame(probe_rows).to_csv(f'{OUT_DIR}/layerwise_probe_results.csv', index=False)

probe_df = pd.DataFrame(probe_rows)
probe_agg = (probe_df.groupby(['encoder', 'task', 'layer'])['acc']
             .agg(['mean', 'std']).reset_index())
probe_agg.round(4).head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

# Esquerda: UPOS — informação satura cedo em ambos os encoders
for enc, color in (('lin_bertimbau', '#4c72b0'), ('bia_bertimbau', '#dd8452')):
    g = probe_agg[(probe_agg.encoder == enc) & (probe_agg.task == 'upos')]
    axes[0].plot(g['layer'], g['mean'], color=color, linewidth=2, marker='o',
                 markersize=5, label=enc)
axes[0].set_title('Probe UPOS por camada')
axes[0].set_xlabel('Camada')
axes[0].set_ylabel('Acurácia do probe')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Direita: HEAD posicional — a dissociação de FORMATO
for enc, color in (('lin_bertimbau', '#4c72b0'), ('bia_bertimbau', '#dd8452')):
    g = probe_agg[(probe_agg.encoder == enc) & (probe_agg.task == 'head')]
    axes[1].plot(g['layer'], g['mean'], color=color, linewidth=2, marker='o',
                 markersize=5, label=enc)
axes[1].set_title('Probe HEAD (posicional) por camada')
axes[1].set_xlabel('Camada')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# O encoder biaffine atinge ~90% de UAS com suas próprias cabeças (§5), mas o probe
# posicional fica em ~26% em todas as camadas: a informação de ligação existe em
# formato RELACIONAL (pares), invisível a probes token-level.

## 7. Early exit implantável para UPOS: encoder truncado + cabeça linear

Consequência prática da §6: se a informação de UPOS satura na camada 3–5, um etiquetador
dedicado não precisa das 12 camadas. Aqui o encoder é **fisicamente truncado**
(`encoder.encoder.layer[:k]` — economia de FLOPs real, ao contrário do logit lens) e uma
`nn.Linear(768, 16)` é treinada sobre as representações congeladas da camada $k$.

Resultados de referência (`artifacts/layerwise/upos_early_exit_benchmark.csv`, RTX 4090):
camada 3 → 98,35% de acurácia com speedup 3,67×; camada 5 → 99,03% com 2,32×
(MTL implantado: 99,22%).

In [ ]:
import time

EXIT_LAYERS = (2, 3, 4, 5, 6, 12)
EXIT_SEEDS = (0, 1, 2)


class UposEarlyExitTagger(nn.Module):
    """Modelo implantável: BERT truncado na camada k + classificador linear de UPOS."""

    def __init__(self, encoder, k, n_tags=len(UPOS_LABELS)):
        super().__init__()
        encoder.encoder.layer = encoder.encoder.layer[:k]   # truncamento físico
        self.bert = encoder
        self.head = nn.Linear(encoder.config.hidden_size, n_tags)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        hs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                       token_type_ids=token_type_ids).last_hidden_state
        return self.head(hs)


def extract_upos_features(encoder, tokenizer, sentences, upos, layers):
    """Features fp16 no 1º subtoken, apenas para as camadas candidatas a exit."""
    feats, golds = [], []
    encoder.eval()
    for i in tqdm(range(len(sentences)), desc='extract'):
        tokens = sentences[i]
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors='pt',
                           padding=True, truncation=True).to(DEVICE)
        with torch.no_grad():
            out = encoder(input_ids=inputs['input_ids'],
                          attention_mask=inputs['attention_mask'],
                          token_type_ids=inputs.get('token_type_ids'),
                          output_hidden_states=True)
            hs = torch.stack([out.hidden_states[l] for l in layers], dim=0)[:, 0]
        word_ids = inputs.word_ids(batch_index=0)
        first_subs, tok_idxs = [], []
        seen = set()
        for pos, w in enumerate(word_ids):
            if w is not None and w not in seen and w < len(tokens):
                seen.add(w)
                first_subs.append(pos)
                tok_idxs.append(w)
        if not first_subs:
            continue
        feats.append(hs[:, torch.tensor(first_subs, device=DEVICE)].half().cpu())
        golds.append(np.array([UPOS_LABELS_TO_IDX[upos[i][t]] for t in tok_idxs]))
    X = torch.cat(feats, dim=1)
    y = torch.tensor(np.concatenate(golds), dtype=torch.long)
    return X, y

In [ ]:
# Treino das cabeças (encoder do modelo linear congelado, split de treino completo)
X_tr, y_tr = extract_upos_features(linear_model.bert, tokenizer,
                                   train_dataset['tokens'], train_dataset['upos'], EXIT_LAYERS)
X_te, y_te = extract_upos_features(linear_model.bert, tokenizer,
                                   test_sentences, test_upos, EXIT_LAYERS)

exit_rows = []
trained_heads = {}
for layer_pos, layer in enumerate(EXIT_LAYERS):
    Xl_tr = X_tr[layer_pos].float().to(DEVICE)
    Xl_te = X_te[layer_pos].float().to(DEVICE)
    for seed in EXIT_SEEDS:
        acc = train_probe(Xl_tr, y_tr.to(DEVICE), Xl_te, y_te.to(DEVICE),
                          len(UPOS_LABELS), seed)
        exit_rows.append({'layer': layer, 'seed': seed, 'upos_acc': acc})
        print(f'L={layer:2d} seed={seed} | upos_acc={acc:.4f}')
    del Xl_tr, Xl_te
    torch.cuda.empty_cache()

exit_df = pd.DataFrame(exit_rows)
exit_df.to_csv(f'{OUT_DIR}/upos_early_exit_results.csv', index=False)
exit_df.groupby('layer')['upos_acc'].agg(['mean', 'std']).round(4)

In [ ]:
# Benchmark de latência real: tagger truncado em k camadas no test set completo
@torch.no_grad()
def benchmark_latency(k, encoded_batches, warmup=5):
    encoder = AutoModel.from_pretrained(LINEAR_MODEL_PATH, add_pooling_layer=False)
    model = UposEarlyExitTagger(encoder, k).to(DEVICE).eval()
    params = sum(p.numel() for p in model.parameters()) / 1e6
    for _ in range(warmup):
        model(**encoded_batches[0])
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for enc in encoded_batches:
        model(**enc)
    torch.cuda.synchronize()
    sec = time.perf_counter() - t0
    del model
    torch.cuda.empty_cache()
    return sec, params


BENCH_BATCH = 32
encoded_batches = [
    tokenizer(test_sentences[i:i + BENCH_BATCH], is_split_into_words=True,
              return_tensors='pt', padding=True, truncation=True).to(DEVICE)
    for i in range(0, len(test_sentences), BENCH_BATCH)]

acc_mean = exit_df.groupby('layer')['upos_acc'].mean()
bench, sec_full = [], None
for k in sorted(EXIT_LAYERS, reverse=True):
    sec, params = benchmark_latency(k, encoded_batches)
    if k == max(EXIT_LAYERS):
        sec_full = sec
    bench.append({'layer': k, 'params_M': round(params, 1),
                  'sec_test_set': round(sec, 3), 'speedup': round(sec_full / sec, 2),
                  'acc_mean': round(acc_mean[k], 4)})
bench_df = pd.DataFrame(bench).sort_values('layer')
bench_df.to_csv(f'{OUT_DIR}/upos_early_exit_benchmark.csv', index=False)
bench_df

## 8. Cabeças congeladas nas camadas de saturação e resultados de referência

O experimento complementar — re-treinar famílias completas de cabeças (linear e biaffine)
sobre o encoder **congelado** do BERTimbau-large (24 camadas) nas camadas de saturação de
cada tarefa (UPOS@10, DEPREL@18, HEAD@23) — está em `../scripts/layerwise_frozen_heads.py`
(resultados em `../artifacts/layerwise/layerwise_frozen_heads_results.csv`). Ele fornece a
prova **causal** da dissociação de formato: da mesma representação congelada, uma cabeça
biaffine re-treinada recupera 91 UAS enquanto uma cabeça posicional extrai no máximo 34.

Os demais scripts da análise (`layerwise_probes.py` com 5 encoders/5 seeds,
`layerwise_mbert_analysis.py`, `layerwise_large_analysis.py`,
`layerwise_independent_runs.py`, `layerwise_comparison_aaai.py` — figuras, tabelas e
bootstrap pareado do artigo) estão em `../scripts/`.

### Resultados de referência (test set Porttinari, 1.683 sentenças)

| Leitura | UPOS | DEPREL | HEAD |
|---|---|---|---|
| Convergência (logit lens, linear) | camada ~8 | ~9 | ~12 |
| Convergência (logit lens, biaffine) | ~6 | ~9 | ~11 |
| Saturação do probe (informação) | ~3 | 8–9 | formato-dependente |
| Early exit na camada 10 (linear) | −0,3 p.p. | −0,5 p.p. | **−48,6 UAS** |
| Exit UPOS dedicado, camada 3 | 98,35% (3,67× mais rápido) | — | — |
| Exit UPOS dedicado, camada 5 | 99,03% (2,32× mais rápido) | — | — |

**Conclusão**: informação, comportamento e formato dão respostas diferentes para "onde a
sintaxe mora". Early exit é seguro para o que satura cedo (tagging) com cabeça calibrada
na camada de saída, e estruturalmente inviável para a predição de ligações sintáticas.